# 04 Statistical Analysis

Formal tests on the cleaned datasets to back the EDA observations.

**Analyses**
1. Correlation matrix across NEA orbital + physical features
2. Welch's t-test: PHA vs non-PHA diameter (log)
3. Welch's t-test: PHA vs non-PHA MOID
4. Pearson correlation: close-approach distance vs velocity
5. Chi-square: size_category × PHA independence
6. Linear trend test: annual close-approach counts vs year


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

close = pd.read_csv(PROCESSED_DIR / 'close_approaches_clean.csv', parse_dates=['close_approach_date'])
nea   = pd.read_csv(PROCESSED_DIR / 'near_earth_asteroids_clean.csv')
print('close:', close.shape, '| nea:', nea.shape)


## 1. Correlation matrix (NEA numeric features)

In [ ]:
numeric_cols = ['h', 'diameter_km', 'albedo', 'e', 'a', 'i', 'q', 'ad', 'per_y', 'moid_au', 'moid_lunar_distances']
corr = nea[numeric_cols].corr(method='pearson')
corr.round(3)


## 2. Welch's t-test — log(diameter) by PHA flag

H0: mean log-diameter is equal between PHA and non-PHA. We use log because diameter is heavily right-skewed.

In [ ]:
pha_diam = np.log(nea.loc[nea['pha'] == True,  'diameter_km'].dropna())
non_diam = np.log(nea.loc[nea['pha'] == False, 'diameter_km'].dropna())

t, p = stats.ttest_ind(pha_diam, non_diam, equal_var=False)
print(f'PHA n={len(pha_diam):,}, mean log(d)={pha_diam.mean():.3f}')
print(f'Non-PHA n={len(non_diam):,}, mean log(d)={non_diam.mean():.3f}')
print(f't = {t:.3f} | p = {p:.3e}')
print('Interpretation:', 'PHAs are significantly larger.' if p < 0.05 and t > 0 else 'No significant difference.')


## 3. Welch's t-test — MOID (lunar distances) by PHA flag

PHAs are defined partly by MOID ≤ 0.05 AU, so we expect a strong effect. This is a sanity check.

In [ ]:
pha_moid = nea.loc[nea['pha'] == True,  'moid_lunar_distances'].dropna()
non_moid = nea.loc[nea['pha'] == False, 'moid_lunar_distances'].dropna()

t, p = stats.ttest_ind(pha_moid, non_moid, equal_var=False)
print(f'PHA     MOID (LD) — median: {pha_moid.median():.2f}, mean: {pha_moid.mean():.2f}')
print(f'Non-PHA MOID (LD) — median: {non_moid.median():.2f}, mean: {non_moid.mean():.2f}')
print(f't = {t:.3f} | p = {p:.3e}')


## 4. Pearson — close-approach distance vs relative velocity

In [ ]:
r, p = stats.pearsonr(close['dist_lunar'], close['velocity_km_s'])
print(f'Pearson r = {r:.4f} | p = {p:.3e}')

rs, ps = stats.spearmanr(close['dist_lunar'], close['velocity_km_s'])
print(f'Spearman ρ = {rs:.4f} | p = {ps:.3e}')


## 5. Chi-square — size_category × PHA independence

In [ ]:
ct = pd.crosstab(nea['size_category'], nea['pha'])
chi2, p, dof, expected = stats.chi2_contingency(ct)
print('Contingency table:')
print(ct)
print(f'\nchi2 = {chi2:.2f} | dof = {dof} | p = {p:.3e}')
print('Interpretation:', 'Size category and PHA flag are NOT independent.' if p < 0.05 else 'No significant association.')


## 6. Linear trend — close approaches per year

In [ ]:
yearly = close.groupby('approach_year').size().rename('approaches').reset_index()
slope, intercept, r, p, se = stats.linregress(yearly['approach_year'], yearly['approaches'])
print(f'slope = {slope:.2f} approaches/year | r = {r:.3f} | p = {p:.3e}')
print('Interpretation:', 'Statistically significant trend.' if p < 0.05 else 'No significant linear trend in approach counts.')
yearly.head()


### Summary

- PHA flag correlates with larger diameter and (by definition) smaller MOID — both highly significant.
- Close-approach distance and velocity show weak correlation — closer passes are not systematically faster or slower.
- Size category and PHA are strongly dependent: larger size buckets over-represent PHAs.
- No meaningful linear trend in annual approach counts — useful context for the dashboard (the signal is in individual events, not volume growth).